# Train Decision Segmenter (OPF) — v5 Label Space

Fine-tune the OpenAI Privacy Filter (1.5B params) as a 21-class judicial decision segmenter.

**Pipeline:** OPF base model detects entities → Haiku refines person roles → heuristic adds sections → merged training data

**Requirements:** GPU runtime (T4 or better, 15GB+ VRAM) + `ANTHROPIC_API_KEY` for Haiku step

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Switch to GPU runtime first!"

In [ ]:
!pip install -q "opf @ git+https://github.com/openai/privacy-filter.git" httpx

In [ ]:
# Download OPF base checkpoint (~2.8GB)
from opf._common.checkpoint_download import ensure_default_checkpoint
ensure_default_checkpoint()

In [ ]:
# Clone repo and get prepared data
!git clone --depth 1 --branch claude/epic-clarke-9uaaQ https://github.com/franklinbaldo/causaganha.git /content/causaganha

In [ ]:
import os
os.chdir("/content/causaganha")
os.environ["PYTHONPATH"] = "/content/causaganha"

!pip install -q ibis-framework[duckdb] structlog pyarrow

# Set your Anthropic API key for the Haiku refinement step
# Get one at https://console.anthropic.com/
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
# === DATA PREPARATION ===
# Option A (RECOMMENDED): OPF bootstrap + Haiku refinement
# Downloads texts from IA, runs OPF base model for entity detection,
# sends person names to Haiku for role classification (parte_autor, nome_advogado, etc.),
# merges with heuristic section labels. ~15-20 min on T4.

# Step 1: Download texts from IA (reuses augment script for downloading only)
!python scripts/augment_segmenter_data.py \
    --target 2000 \
    --max-per-tribunal 150 \
    --output-dir /content/models/decision_segmenter \
    --n-items 10

In [ ]:
# Step 2: Bootstrap high-quality labels using OPF + Haiku
# OPF detects entities (person, email, phone, address, date, account_number)
# Auto-maps unambiguous labels (email→email, phone→telefone, etc.)
# Sends person names to Haiku for role classification (~$1-2 for 2000 texts)
# Merges with heuristic structural sections (sec_cabecalho, sec_dispositivo, etc.)
#
# To skip Haiku (no API key), add --skip-haiku flag (uses only OPF + heuristic)

!python scripts/bootstrap_training_corpus.py \
    --input /content/models/decision_segmenter/train.jsonl \
    --output-dir /content/models/decision_segmenter_v5 \
    --target 2000 \
    --opf-batch-size 4 \
    --haiku-batch-size 40

In [ ]:
# Train with OPF on T4 (batch_size=1 + n-ctx=512 to fit in 15GB VRAM)
# On A100 (40GB): increase to --batch-size 4 --n-ctx 1024
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Uses v5 label space (21 classes) from bootstrap output
MODEL_DIR = "/content/models/decision_segmenter_v5"

!python -m opf train {MODEL_DIR}/train.jsonl \
    --validation-dataset {MODEL_DIR}/val.jsonl \
    --label-space-json {MODEL_DIR}/label_space.json \
    --output-dir {MODEL_DIR}/best \
    --device cuda \
    --epochs 3 \
    --batch-size 1 \
    --n-ctx 512

In [ ]:
# Evaluate on test set (eval reads label space from the checkpoint)
MODEL_DIR = "/content/models/decision_segmenter_v5"

!python -m opf eval {MODEL_DIR}/test.jsonl \
    --checkpoint {MODEL_DIR}/best \
    --device cuda \
    --per-class \
    --metrics-out {MODEL_DIR}/test_metrics.json

In [ ]:
import json

MODEL_DIR = "/content/models/decision_segmenter_v5"
metrics = json.load(open(f"{MODEL_DIR}/test_metrics.json"))

print("=" * 60)
print("TEST METRICS (v5 label space)")
print("=" * 60)
macro = metrics.get("macro avg", {})
print(f"Macro F1: {macro.get('f1-score', 0):.3f}")
disp = metrics.get("sec_dispositivo", {})
print(f"sec_dispositivo F1: {disp.get('f1-score', 0):.3f}")
print()
for k, v in metrics.items():
    if isinstance(v, dict) and "f1-score" in v and k not in ("macro avg", "weighted avg", "micro avg"):
        print(f"  {k:<22} P={v.get('precision',0):.2f}  R={v.get('recall',0):.2f}  F1={v.get('f1-score',0):.2f}  n={v.get('support',0)}")

In [ ]:
# Download trained model (save to Google Drive or download directly)
MODEL_DIR = "/content/models/decision_segmenter_v5"
!tar -czf /content/decision_segmenter_v5.tar.gz -C {MODEL_DIR}/best .
print(f"Model saved: /content/decision_segmenter_v5.tar.gz")

# Optional: mount Drive and copy
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/decision_segmenter_v5.tar.gz /content/drive/MyDrive/
# !cp {MODEL_DIR}/test_metrics.json /content/drive/MyDrive/